# CGAP–WESAD: Calibration Evaluation and Leakage/Data Audit for a Q1 Manuscript

This notebook is a **reproducible audit template**, not a claim that the CGAP results have passed these checks. It performs:

1. Modelling-table integrity checks (participants, labels, missing/non-finite values, duplicate rows, suspicious metadata/features).
2. Prediction-file integrity checks (class-probability columns, row counts, valid probability ranges/sums, missing values, duplicates).
3. Multiclass calibration metrics: multiclass Brier score, log loss, top-label expected calibration error (ECE), classwise one-vs-rest ECE, and reliability diagrams.
4. Participant-cluster bootstrap confidence intervals for calibration metrics.
5. Optional post-hoc temperature scaling fitted **only on validation predictions**, then evaluated once on held-out test predictions.
6. A machine-readable audit report and CSV/PNG outputs.

## Important prerequisites

- Calibration requires **per-window predicted probabilities**, not only accuracy/F1 summary rows.
- Create two CSV files from the model pipeline: `validation_predictions.csv` and `test_predictions.csv`.
- Each file should contain: `subject`, `y_true`, and one probability column per class (`p_baseline`, `p_stress`, `p_amusement`). Optional columns: `model`, `seed`, `window_idx`, `fold`.
- Use the same class order and label mapping in both files. Probabilities must be predictions generated before the true label is used for any online update at that time step.
- For the current CGAP evaluation, online updates use the revealed test-window label **after prediction**. Report this as *labelled-feedback replay*, not label-free adaptation.
- Calibration must be assessed on held-out data. Do not fit calibration parameters on the test set.
- This notebook cannot independently prove the absence of leakage from CSVs alone. It creates auditable checks and flags requiring inspection of upstream feature-generation code.



## 1. Install/import packages and configure paths

In [ ]:
%pip -q install numpy pandas scipy scikit-learn matplotlib seaborn

from pathlib import Path
import json, re, warnings, hashlib, platform, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.special import softmax
from sklearn.metrics import log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit

# Edit these paths to match your Google Drive.
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/DataCon/Project 6 Wesad')
DATA_DIR = PROJECT_DIR
VALIDATION_PRED_FILE = PROJECT_DIR / 'validation_predictions.csv'
TEST_PRED_FILE = PROJECT_DIR / 'test_predictions.csv'
OUTPUT_DIR = PROJECT_DIR / 'calibration_audit_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBJECT_COL_CANDIDATES = ['subject', 'participant', 'participant_id', 'user_id']
LABEL_COL_CANDIDATES = ['label', 'state', 'target', 'class', 'y']
WINDOW_COL_CANDIDATES = ['window_idx', 'window_index', 'window', 'timestamp', 'start_time']
CLASS_LABELS = ['baseline', 'stress', 'amusement']
PROB_COLS = ['p_baseline', 'p_stress', 'p_amusement']
N_BOOT = 10000
RANDOM_STATE = 42
ECE_BINS = 15

print('Python:', sys.version)
print('Platform:', platform.platform())
print('NumPy:', np.__version__, '| pandas:', pd.__version__)
print('Project directory:', PROJECT_DIR)
print('Output directory:', OUTPUT_DIR)


## 2. Audit source modelling CSV files

In [ ]:
def first_col(candidates, columns):
    return next((c for c in candidates if c in columns), None)

source_files = sorted(
    p for p in DATA_DIR.glob('S*_windows.csv')
    if re.fullmatch(r'S\\d+_windows', p.stem)
)
audit_items = []

def add_audit(check, status, details):
    audit_items.append({'check': check, 'status': status, 'details': str(details)})

if not source_files:
    add_audit('source_files_found', 'WARNING', f'No S*_windows.csv files found in {DATA_DIR}')
    source_df = None
else:
    parts = []
    for path in source_files:
        tmp = pd.read_csv(path)
        if 'subject' not in tmp.columns:
            tmp['subject'] = path.stem.split('_')[0]
        tmp['_source_file'] = path.name
        parts.append(tmp)
    source_df = pd.concat(parts, ignore_index=True)

    subj_col = first_col(SUBJECT_COL_CANDIDATES, source_df.columns)
    label_col = first_col(LABEL_COL_CANDIDATES, source_df.columns)
    window_col = first_col(WINDOW_COL_CANDIDATES, source_df.columns)

    add_audit('source_file_count', 'PASS' if len(source_files) == 15 else 'WARNING',
              f'{len(source_files)} files found; expected 15 for S1-S15')
    add_audit('row_count', 'INFO', len(source_df))
    add_audit('column_count', 'INFO', len(source_df.columns))
    add_audit('subject_column_found', 'PASS' if subj_col else 'FAIL', subj_col)
    add_audit('label_column_found', 'PASS' if label_col else 'FAIL', label_col)

    if subj_col:
        subjects = sorted(source_df[subj_col].astype(str).unique())
        add_audit('unique_subject_count', 'PASS' if len(subjects) == 15 else 'WARNING',
                  f'{len(subjects)} unique subjects: {subjects}')
        subject_counts = source_df.groupby(source_df[subj_col].astype(str)).size().rename('rows').reset_index()
        subject_counts.to_csv(OUTPUT_DIR / 'source_rows_per_subject.csv', index=False)
    if label_col:
        label_counts = source_df[label_col].value_counts(dropna=False).rename_axis('label').reset_index(name='count')
        label_counts.to_csv(OUTPUT_DIR / 'source_label_counts.csv', index=False)
        add_audit('observed_labels', 'INFO', source_df[label_col].value_counts(dropna=False).to_dict())
        add_audit('label_missing', 'PASS' if source_df[label_col].isna().sum() == 0 else 'FAIL',
                  int(source_df[label_col].isna().sum()))

    numeric_cols = source_df.select_dtypes(include=[np.number]).columns.tolist()
    feature_like = [c for c in numeric_cols if c not in [window_col]]
    missing_counts = source_df[feature_like].isna().sum().sort_values(ascending=False)
    nonfinite_counts = pd.Series({
        c: int((~np.isfinite(pd.to_numeric(source_df[c], errors='coerce').to_numpy(dtype=float, na_value=np.nan))).sum())
        for c in feature_like
    }).sort_values(ascending=False)
    missing_counts.rename('missing_count').to_csv(OUTPUT_DIR / 'source_missing_counts.csv')
    nonfinite_counts.rename('nonfinite_count').to_csv(OUTPUT_DIR / 'source_nonfinite_counts.csv')

    add_audit('rows_with_any_missing_numeric', 'INFO',
              int(source_df[feature_like].isna().any(axis=1).sum()) if feature_like else 'no numeric columns')
    add_audit('numeric_columns_with_missing', 'INFO',
              missing_counts[missing_counts > 0].to_dict())
    add_audit('numeric_columns_with_nonfinite', 'INFO',
              nonfinite_counts[nonfinite_counts > 0].to_dict())

    # Exact row duplicates are a warning; identical features across different windows need manual review.
    exact_dup_count = int(source_df.drop(columns=['_source_file'], errors='ignore').duplicated().sum())
    add_audit('exact_duplicate_rows', 'PASS' if exact_dup_count == 0 else 'WARNING', exact_dup_count)

    # Detect likely target/metadata leakage by name; manual review remains necessary.
    suspicious = [c for c in source_df.columns if re.search(
        r'(^|_)(label|target|class|state|subject|participant|user|id|outcome|future|next|stress_label)(_|$)',
        str(c), flags=re.I
    )]
    add_audit('metadata_or_target_like_columns_for_review', 'INFO', suspicious)

    print('Source shape:', source_df.shape)
    print('Source files:', len(source_files))
    display(source_df.head())

display(pd.DataFrame(audit_items))


## 3. Load per-window validation and test probabilities

In [ ]:
for p in [VALIDATION_PRED_FILE, TEST_PRED_FILE]:
    if not p.exists():
        raise FileNotFoundError(
            f'Missing {p}. Export per-window predicted probabilities from the CGAP notebook first. '
            'Summary-only accuracy/F1 CSVs are insufficient for calibration analysis.'
        )

val = pd.read_csv(VALIDATION_PRED_FILE)
test = pd.read_csv(TEST_PRED_FILE)

required = {'y_true', *PROB_COLS}
for name, frame in [('validation', val), ('test', test)]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{name} predictions missing columns: {sorted(missing)}')
    if frame[list(required)].isna().any().any():
        raise ValueError(f'{name} predictions contain missing labels/probabilities.')
    if not np.isfinite(frame[PROB_COLS].to_numpy(dtype=float)).all():
        raise ValueError(f'{name} probability matrix contains NaN/Inf.')

# Normalize labels to lowercase strings; edit this mapping if your saved labels use different names.
def normalize_label(x):
    s = str(x).strip().lower()
    # Do NOT assume numeric IDs 0/1/2 correspond to a particular class.
    # If predictions contain numeric labels, define their mapping explicitly below.
    aliases = {'neutral':'baseline', 'rest':'baseline', 'fun':'amusement'}
    return aliases.get(s, s)

# If y_true is numeric (e.g., 0/1/2), explicitly map IDs using the exact
# LabelEncoder.classes_ order saved by your CGAP training run. Example ONLY:
# NUMERIC_LABEL_MAP = {0: 'amusement', 1: 'baseline', 2: 'stress'}
# Do not use this example unless it matches your actual training encoder.
if pd.api.types.is_numeric_dtype(val['y_true']) or pd.api.types.is_numeric_dtype(test['y_true']):
    raise ValueError(
        'y_true contains numeric class IDs. Map them to the exact trained class names '
        'using the saved LabelEncoder.classes_ order before running calibration.'
    )

for frame in [val, test]:
    frame['y_true'] = frame['y_true'].map(normalize_label)

unknown_labels = (set(val['y_true']) | set(test['y_true'])) - set(CLASS_LABELS)
if unknown_labels:
    raise ValueError(
        f'Unexpected labels: {unknown_labels}. Update CLASS_LABELS/label mapping to match the trained model.'
    )

for name, frame in [('validation', val), ('test', test)]:
    P = frame[PROB_COLS].to_numpy(dtype=float)
    in_range = bool(((P >= 0) & (P <= 1)).all())
    row_sums = P.sum(axis=1)
    add_audit(f'{name}_probabilities_in_[0,1]', 'PASS' if in_range else 'FAIL', in_range)
    add_audit(f'{name}_probability_sum_max_abs_error', 'INFO', float(np.max(np.abs(row_sums - 1))))
    if not in_range or not np.allclose(row_sums, 1, atol=1e-4):
        raise ValueError(f'{name} probabilities are invalid or do not sum to 1. Do not silently normalize them.')

print('Validation rows:', len(val), '| Test rows:', len(test))
print('Validation columns:', val.columns.tolist())
print('Test columns:', test.columns.tolist())
display(val.head())


## 4. Audit prediction identities, participant overlap, and probability integrity

In [ ]:
def audit_predictions(frame, split_name):
    subject_col = first_col(SUBJECT_COL_CANDIDATES, frame.columns)
    window_col = first_col(WINDOW_COL_CANDIDATES, frame.columns)
    add_audit(f'{split_name}_prediction_row_count', 'INFO', len(frame))
    add_audit(f'{split_name}_subjects_available', 'INFO',
              sorted(frame[subject_col].astype(str).unique().tolist()) if subject_col else 'no subject column')
    add_audit(f'{split_name}_exact_duplicate_prediction_rows', 
              'PASS' if frame.duplicated().sum() == 0 else 'WARNING', int(frame.duplicated().sum()))
    if subject_col and window_col:
        dup_keys = int(frame.duplicated([subject_col, window_col]).sum())
        add_audit(f'{split_name}_duplicate_subject_window_keys',
                  'PASS' if dup_keys == 0 else 'FAIL', dup_keys)
    # Class coverage
    counts = frame['y_true'].value_counts().reindex(CLASS_LABELS, fill_value=0)
    add_audit(f'{split_name}_class_counts', 'INFO', counts.to_dict())
    return subject_col

val_subject_col = audit_predictions(val, 'validation')
test_subject_col = audit_predictions(test, 'test')

if val_subject_col and test_subject_col:
    val_subjects = set(val[val_subject_col].astype(str))
    test_subjects = set(test[test_subject_col].astype(str))
    overlap = sorted(val_subjects & test_subjects)
    add_audit('validation_test_subject_overlap',
              'PASS' if not overlap else 'WARNING',
              f'overlap={overlap}; expected disjoint participants for strict participant-held-out calibration.')
    if overlap:
        warnings.warn('Validation and test contain the same participant(s). Ensure calibration and evaluation are grouped correctly.')
else:
    add_audit('validation_test_subject_overlap', 'NOT CHECKED',
              'Subject column absent; participant-level split audit cannot be verified.')

display(pd.DataFrame(audit_items))


## 5. Define multiclass calibration metrics

In [ ]:
def labels_to_onehot(y, labels=CLASS_LABELS):
    lookup = {label: i for i, label in enumerate(labels)}
    y_idx = np.array([lookup[v] for v in y], dtype=int)
    Y = np.zeros((len(y_idx), len(labels)), dtype=float)
    Y[np.arange(len(y_idx)), y_idx] = 1.0
    return y_idx, Y

def multiclass_brier(y, P, labels=CLASS_LABELS):
    _, Y = labels_to_onehot(y, labels)
    return float(np.mean(np.sum((P - Y) ** 2, axis=1)))

def top_label_ece(y, P, labels=CLASS_LABELS, n_bins=15):
    y_idx, _ = labels_to_onehot(y, labels)
    conf = P.max(axis=1)
    pred = P.argmax(axis=1)
    correct = (pred == y_idx).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    rows = []
    for i in range(n_bins):
        # Last bin includes confidence == 1.
        mask = (conf >= edges[i]) & ((conf < edges[i+1]) if i < n_bins-1 else (conf <= edges[i+1]))
        n = int(mask.sum())
        if n:
            acc_bin = float(correct[mask].mean())
            conf_bin = float(conf[mask].mean())
            ece += (n / len(conf)) * abs(acc_bin - conf_bin)
        else:
            acc_bin, conf_bin = np.nan, np.nan
        rows.append({'bin': i, 'lower': edges[i], 'upper': edges[i+1], 'count': n,
                     'mean_confidence': conf_bin, 'accuracy': acc_bin})
    return float(ece), pd.DataFrame(rows)

def classwise_ece(y, P, labels=CLASS_LABELS, n_bins=15):
    _, Y = labels_to_onehot(y, labels)
    rows = []
    edges = np.linspace(0, 1, n_bins + 1)
    for k, cls in enumerate(labels):
        prob = P[:, k]
        outcome = Y[:, k]
        ece = 0.0
        for i in range(n_bins):
            mask = (prob >= edges[i]) & ((prob < edges[i+1]) if i < n_bins-1 else (prob <= edges[i+1]))
            if mask.any():
                ece += (mask.sum()/len(prob)) * abs(float(outcome[mask].mean()) - float(prob[mask].mean()))
        rows.append({'class': cls, 'one_vs_rest_ECE': float(ece)})
    return pd.DataFrame(rows)

def calibration_metrics(frame, P, model_name):
    y = frame['y_true'].tolist()
    y_idx, _ = labels_to_onehot(y)
    pred = P.argmax(axis=1)
    ece, bins = top_label_ece(y, P, n_bins=ECE_BINS)
    return {
        'model': model_name,
        'n_windows': len(frame),
        'accuracy': float(np.mean(pred == y_idx)),
        'multiclass_brier': multiclass_brier(y, P),
        'log_loss': float(log_loss(y_idx, P, labels=np.arange(len(CLASS_LABELS)))),
        'top_label_ECE': ece,
        'mean_confidence': float(P.max(axis=1).mean()),
        'empirical_accuracy': float(np.mean(pred == y_idx))
    }, bins

P_val = val[PROB_COLS].to_numpy(float)
P_test = test[PROB_COLS].to_numpy(float)

val_metrics, val_bins = calibration_metrics(val, P_val, 'CGAP_raw_validation')
test_metrics, test_bins = calibration_metrics(test, P_test, 'CGAP_raw_test')
metrics_df = pd.DataFrame([val_metrics, test_metrics])
display(metrics_df)
metrics_df.to_csv(OUTPUT_DIR / 'calibration_metrics_raw.csv', index=False)
classwise_ece(test['y_true'].tolist(), P_test).to_csv(
    OUTPUT_DIR / 'calibration_classwise_ece_raw_test.csv', index=False
)


## 6. Plot reliability diagrams and confidence distribution

In [ ]:
def plot_reliability(frame, P, tag):
    y_idx, _ = labels_to_onehot(frame['y_true'].tolist())
    pred = P.argmax(axis=1)
    conf = P.max(axis=1)
    correct = (pred == y_idx).astype(float)
    ece, bins = top_label_ece(frame['y_true'].tolist(), P, n_bins=ECE_BINS)
    bins.to_csv(OUTPUT_DIR / f'reliability_bins_{tag}.csv', index=False)

    valid = bins['count'] > 0
    plt.figure(figsize=(6.5, 6))
    plt.plot([0, 1], [0, 1], linestyle='--', label='Perfect calibration')
    plt.plot(bins.loc[valid, 'mean_confidence'], bins.loc[valid, 'accuracy'],
             marker='o', label=f'Observed (ECE={ece:.3f})')
    plt.xlabel('Mean predicted confidence')
    plt.ylabel('Empirical accuracy')
    plt.title(f'Top-label reliability diagram — {tag}')
    plt.xlim(0, 1); plt.ylim(0, 1)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'reliability_{tag}.png', dpi=300, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(7, 4))
    sns.histplot(conf, bins=20, stat='probability')
    plt.xlabel('Maximum predicted class probability')
    plt.ylabel('Proportion of windows')
    plt.title(f'Prediction confidence distribution — {tag}')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'confidence_{tag}.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_reliability(val, P_val, 'validation_raw')
plot_reliability(test, P_test, 'test_raw')


## 7. Participant-cluster bootstrap confidence intervals

In [ ]:
def metric_vector(frame, P):
    y_idx, _ = labels_to_onehot(frame['y_true'].tolist())
    pred = P.argmax(axis=1)
    ece, _ = top_label_ece(frame['y_true'].tolist(), P, n_bins=ECE_BINS)
    return {
        'accuracy': float((pred == y_idx).mean()),
        'multiclass_brier': multiclass_brier(frame['y_true'].tolist(), P),
        'log_loss': float(log_loss(y_idx, P, labels=np.arange(len(CLASS_LABELS)))),
        'top_label_ECE': float(ece)
    }

def cluster_bootstrap_ci(frame, P, subject_col, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    if not subject_col or subject_col not in frame.columns:
        warnings.warn('No participant column; falling back to row bootstrap. This is weaker and should be disclosed.')
        groups = [np.array([i]) for i in range(len(frame))]
    else:
        groups = [np.flatnonzero(frame[subject_col].astype(str).to_numpy() == s)
                  for s in frame[subject_col].astype(str).unique()]
    if len(groups) < 2:
        raise ValueError('Need at least two independent participants/clusters for cluster bootstrap.')
    results = {m: [] for m in ['accuracy','multiclass_brier','log_loss','top_label_ECE']}
    for b in range(n_boot):
        chosen = rng.integers(0, len(groups), size=len(groups))
        idx = np.concatenate([groups[i] for i in chosen])
        boot_frame = frame.iloc[idx].reset_index(drop=True)
        boot_P = P[idx]
        vals = metric_vector(boot_frame, boot_P)
        for m, v in vals.items():
            results[m].append(v)
    point = metric_vector(frame, P)
    out = []
    for metric, arr in results.items():
        out.append({'metric': metric, 'estimate': point[metric],
                    'ci95_lower': float(np.quantile(arr, .025)),
                    'ci95_upper': float(np.quantile(arr, .975)),
                    'bootstrap_replicates': n_boot,
                    'independent_clusters': len(groups)})
    return pd.DataFrame(out)

test_subject_col = first_col(SUBJECT_COL_CANDIDATES, test.columns)
boot_ci = cluster_bootstrap_ci(test, P_test, test_subject_col, n_boot=N_BOOT, seed=RANDOM_STATE)
display(boot_ci)
boot_ci.to_csv(OUTPUT_DIR / 'calibration_test_cluster_bootstrap_ci.csv', index=False)


## 8. Optional temperature scaling — fit on validation, evaluate on test only

Temperature scaling learns one scalar temperature from **validation** probabilities. The test set is not used for fitting. A temperature above 1 generally softens predictions; below 1 sharpens them. This implementation optimizes multiclass validation negative log-likelihood over a one-dimensional temperature grid. It is intentionally simple and reproducible. Do not present calibrated test metrics as a gain unless the test set remained untouched during selection.

In [ ]:
def temperature_scale(P, T):
    P = np.clip(np.asarray(P, dtype=float), 1e-12, 1.0)
    logits = np.log(P)
    return softmax(logits / T, axis=1)

y_val_idx, _ = labels_to_onehot(val['y_true'].tolist())
temperatures = np.exp(np.linspace(np.log(0.25), np.log(4.0), 401))
val_nlls = []
for T in temperatures:
    Pt = temperature_scale(P_val, T)
    val_nlls.append(log_loss(y_val_idx, Pt, labels=np.arange(len(CLASS_LABELS))))
best_T = float(temperatures[int(np.argmin(val_nlls))])
print('Selected temperature using validation only:', best_T)
print('Validation NLL before:', log_loss(y_val_idx, P_val, labels=np.arange(len(CLASS_LABELS))))
print('Validation NLL after:', min(val_nlls))

P_test_temp = temperature_scale(P_test, best_T)
raw_test_m, _ = calibration_metrics(test, P_test, 'CGAP_raw_test')
temp_test_m, temp_bins = calibration_metrics(test, P_test_temp, 'CGAP_temperature_scaled_test')
calibrated_metrics = pd.DataFrame([raw_test_m, temp_test_m])
display(calibrated_metrics)
calibrated_metrics.to_csv(OUTPUT_DIR / 'calibration_metrics_test_raw_vs_temperature.csv', index=False)

plot_reliability(test, P_test_temp, 'test_temperature_scaled')
with open(OUTPUT_DIR / 'temperature_scaling.json', 'w') as f:
    json.dump({'temperature_selected_on_validation': best_T,
               'validation_nll_raw': float(log_loss(y_val_idx, P_val, labels=np.arange(len(CLASS_LABELS)))),
               'validation_nll_temperature_scaled': float(min(val_nlls)),
               'test_used_for_temperature_fitting': False}, f, indent=2)


## 9. Probability-level audit and model/fold consistency

In [ ]:
# If prediction files include model/seed/fold metadata, summarize their coverage.
for name, frame in [('validation', val), ('test', test)]:
    for col in ['model', 'seed', 'fold']:
        if col in frame.columns:
            add_audit(f'{name}_{col}_values', 'INFO', sorted(frame[col].astype(str).unique().tolist()))
    if 'model' in frame.columns and frame['model'].nunique() > 1:
        add_audit(f'{name}_model_mixed_in_single_file', 'WARNING',
                  'File contains multiple model names; analyze each model separately.')

# Check whether validation/test rows appear to be exact duplicates across splits.
key_candidates = [c for c in ['subject', 'participant', 'participant_id', 'window_idx', 'window_index', 'timestamp'] 
                  if c in val.columns and c in test.columns]
if key_candidates:
    val_keys = set(map(tuple, val[key_candidates].astype(str).to_numpy()))
    test_keys = set(map(tuple, test[key_candidates].astype(str).to_numpy()))
    common_keys = len(val_keys & test_keys)
    add_audit('validation_test_shared_row_keys',
              'PASS' if common_keys == 0 else 'WARNING',
              f'{common_keys} shared key tuples using columns {key_candidates}')
else:
    add_audit('validation_test_shared_row_keys', 'NOT CHECKED',
              'No common subject/window key columns available.')

audit_df = pd.DataFrame(audit_items)
audit_df.to_csv(OUTPUT_DIR / 'calibration_data_audit_report.csv', index=False)
display(audit_df)


## 10. Export concise Q1 reporting tables and run manifest

In [ ]:
# Compact table intended as an analysis output, not a prefilled manuscript claim.
raw_test = metric_vector(test, P_test)
temp_test = metric_vector(test, P_test_temp)
report_table = pd.DataFrame([
    {'condition': 'CGAP raw probabilities', **raw_test},
    {'condition': 'CGAP temperature-scaled (T fit on validation)', **temp_test}
])
report_table.to_csv(OUTPUT_DIR / 'q1_calibration_summary_table.csv', index=False)

manifest = {
    'project': 'CGAP-WESAD calibration and audit',
    'validation_file': str(VALIDATION_PRED_FILE),
    'test_file': str(TEST_PRED_FILE),
    'validation_rows': int(len(val)),
    'test_rows': int(len(test)),
    'class_labels_order': CLASS_LABELS,
    'probability_columns_order': PROB_COLS,
    'ece_bins': ECE_BINS,
    'bootstrap_replicates': N_BOOT,
    'random_state': RANDOM_STATE,
    'temperature_fit_split': 'validation only',
    'test_used_for_calibration_fit': False,
    'warning': 'This report audits supplied files; it does not prove upstream leakage-free feature construction.'
}
with open(OUTPUT_DIR / 'run_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print('Saved outputs in:', OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)


## 11. Q1 journal reporting checklist

Before submission, confirm all items below and report actual results only:

- [ ] Calibration is computed from held-out per-window probabilities, not training probabilities.
- [ ] Temperature scaling or any other calibrator is fitted using validation data only; test data are used once for final evaluation.
- [ ] Report multiclass Brier score and log loss; report ECE with bin count and binning method.
- [ ] Show reliability diagrams and classwise calibration results for baseline, stress, and amusement.
- [ ] Give participant-cluster bootstrap 95% confidence intervals, not only window-level intervals.
- [ ] Report sample size, participant count, class prevalence, probability column order, calibration procedure, and random seeds.
- [ ] Verify that the saved probability for each test window was generated before the true label was used for the online update.
- [ ] Re-run calibration across all outer LOSO folds and seeds before making a full-study claim. A single fold is only a diagnostic.
- [ ] Audit feature-generation code for overlapping windows, label-derived features, future-window summaries, participant leakage, and preprocessing fitted before splitting.
- [ ] Reconcile the modelling table with the EDA dataset; do not report statistics from a different data export as if it were the modelling dataset.
- [ ] Describe online adaptation as **labelled-feedback replay** if test labels are used after each prediction.
- [ ] Do not claim that the audit proves absence of leakage. Report the checks performed and any unresolved limitations.

### Recommended manuscript wording

“Probability calibration was evaluated on held-out predictions using multiclass Brier score, negative log-likelihood, top-label expected calibration error, classwise one-vs-rest ECE, and reliability diagrams. Confidence intervals were estimated by resampling participants as clusters. Temperature scaling was fitted on validation predictions only and then evaluated on the held-out test predictions. Data-integrity checks covered class labels, participant representation, duplicate identifiers, missing/non-finite values, probability bounds and row sums, and validation–test overlap. These checks were complemented by manual inspection of feature-generation and sequential-update code; file-level checks alone cannot establish the absence of all leakage.”

Replace this wording if your actual execution differs. Do not insert any metrics until the notebook has run successfully on the final saved predictions.
